In [1]:
import random
import json
import sys
import numpy as np

from typing import Dict, Any, Sequence, Optional 
from importnb import Notebook

with Notebook():
    from LabTrajectory import simulate_viewport_with_tiles

# sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
from Common.Utils import zipf, poisson_per_users


In [ ]:
class UserTileRequestEvents:
    def __init__(
        self,
        n_users: int = 1,
        step_size: float = 5.0,
        alpha: float = 1.0,
        n_videos: int = 100,
        n_gops: int = 60,
        n_layers: int = 1,
        n_tiles: int = 4,
        n: int = 4,
        arrival_rate: float = 10.0,
        users_viewport_tiles: Optional[Sequence[Any]] = [],
        requested_videos: Optional[Sequence[Any]] = [],
        users_arrivals: Optional[Sequence[Any]] = []
    ):
        self.step_count = 0
        self.users_step_count = [0 for _ in range(n_users)]

        self.n_users = n_users
        self.users_attended = 0

        self.step_size = step_size
        self.alpha = alpha
        self.n_videos = n_videos
        self.n_gops = n_gops
        self.n_layers = n_layers
        self.n_tiles = n_tiles
        self.n = n
        self.users_viewport_tiles = users_viewport_tiles
        self.requested_videos = requested_videos
        self.users_arrivals = users_arrivals
        self.arrival_rate = arrival_rate

    def gen_request_for_user(
        self, 
        u_id: int, 
        p_id: int,
        by_video
    ) -> Dict[str, Any]:
        video = self.requested_videos[u_id]
        step = self.users_step_count[u_id]
 
        tiles = []
        for y_idx in range(self.n):
            for x_idx in range(self.n):
                tile = {
                    "tile": y_idx * self.n + x_idx,
                    "layer": 0,
                    "size": 2e6 / self.n_tiles,
                    "events": {
                        "alpha_p_u": 1 if by_video[video][0][y_idx * self.n + x_idx] > 0 else 0, 
                        "alpha_M_u": 0, 
                        "alpha_C_u": 0 if by_video[video][0][y_idx * self.n + x_idx] > 0 else 1, 
                        "beta_p_u": 0, 
                        "beta_M_u": 0   
                    }
                }
                tiles.append(tile)

        for x, y in self.users_viewport_tiles[u_id][step]:
            if x < 0 or x >= self.n or y < 0 or y >= self.n:
                continue

            tile = {
                "tile": y * self.n + x,
                "layer": 1,
                "size": 15e6 / self.n_tiles,
                "events": {
                    "alpha_p_u": 1 if by_video[video][1][y * self.n + x] > 0 else 0, 
                    "alpha_M_u": 0, 
                    "alpha_C_u": 0 if by_video[video][1][y * self.n + x] > 0 else 1, 
                    "beta_p_u": 0, 
                    "beta_M_u": 0
                }
            }
            tiles.append(tile)

        self.users_step_count[u_id] += 1
        if self.users_step_count[u_id] >= self.n_gops:
            self.users_attended += 1
        
        return {
            "seq": step,
            "u": u_id,
            "p": p_id, 
            "video": video,
            "tiles": tiles
        }

    def step(self, by_video):
        p = 0  # assuming DU id 0 for simplicity
        reqs = [
            self.gen_request_for_user(i, p, by_video)
            for i in range(self.n_users) if self.users_arrivals[i] <= self.step_count and self.users_step_count[i] < self.n_gops
        ]

        self.step_count += 1
        return reqs

    def reset(self, **kwargs):
        self.step_count = 0
        self.users_step_count = [0 for _ in range(self.n_users)]
        self.users_attended = 0

        self.users_arrivals = poisson_per_users(
            total_users=self.n_users,
            rate_per_minute=self.arrival_rate
        )
        
        self.users_viewport_tiles = []
        for _ in range(self.n_users):
            _, _, viewport_tiles, _ = simulate_viewport_with_tiles(
                num_steps=self.n_gops,
                n=self.n,
                fov_yaw=90,
                fov_pitch=50,
                damping=0.99,
                step_size=self.step_size,
                start_yaw=180,
                start_pitch=0
            )
            self.users_viewport_tiles.append(viewport_tiles)
        
        self.requested_videos = zipf(
            samples=self.n_users, 
            total_videos=self.n_videos, 
            alpha=self.alpha
        )
        self.requested_videos = [int(v-1) for v in self.requested_videos]

        info = {
            "viewport_tiles": self.users_viewport_tiles,
            "requested_videos": self.requested_videos,
            "users_arrivals": self.users_arrivals
        }
        
        return None, info

In [3]:
# Validation helpers
def validate_request_struct(req, num_tiles: int) -> bool:
    assert set([
        'seq','u','p','video','tiles'
    ]).issubset(req.keys()), 'Missing top-level keys'
    
    tiles = req['tiles']
    
    assert len([
        t for t in tiles if t['layer'] == 0
    ]) == num_tiles, 'Base layer tiles count mismatch'
    
    for t in tiles:
        for k in ['tile','layer','size','events']:
            assert k in t, f'Missing tile key {k}'
        
        ev = t['events']
        
        for ek in ['alpha_p_u','alpha_M_u','alpha_C_u','beta_p_u','beta_M_u']:
            assert ek in ev, f'Missing event key {ek}'
    
    return True

In [ ]:
if __name__ == '__main__':
    steps_to_run = 3
    n_users = 2
    step_size = 5.0
    alpha = 1.0
    n_videos = 2
    n_gops = 60
    n_layers = 2    # base + enhancement
    n = 4           # grid dimension (n x n)
    n_tiles = n * n # total tiles

    _, _, viewport_tiles, _ = simulate_viewport_with_tiles(
        num_steps=n_gops,
        n=n,
        fov_yaw=90,
        fov_pitch=50,
        damping=0.99,
        step_size=step_size,
        start_yaw=180,
        start_pitch=0
    )

    print('='*5, 'UserTileRequestEvents Test Harness', '='*5)
    print(f'Grid: {n}x{n} -> {n_tiles} tiles | Users: {n_users} | Layers: {n_layers}')
    print('Running steps...')

    user_env = UserTileRequestEvents(
        n=n,
        n_users=n_users,
        step_size=step_size,
        alpha=alpha,
        n_videos=n_videos,
        n_layers=n_layers,
        n_tiles=n_tiles
    )

    user_env.reset(
        step_size=step_size,
        alpha=alpha,
        total_videos=n_videos,
        n_gops=n_gops
    )

    all_requests = []
    for step in range(steps_to_run):
        base_layer = np.ones(n_tiles, dtype=int)

        enh_layer_v0 = np.zeros(n_tiles, dtype=int)
        enh_layer_v1 = np.zeros(n_tiles, dtype=int)

        random_tile_index = random.choice(range(n_tiles))
        enh_layer_v0[random_tile_index] = 1

        random_tile_index = random.choice(range(n_tiles))
        enh_layer_v1[random_tile_index] = 1

        by_video = {
            0: [base_layer, enh_layer_v0],
            1: [base_layer, enh_layer_v1]
        }

        reqs = user_env.step(by_video=by_video)
        print(f' Step {step}: got {len(reqs)} user request bundles')

        for r in reqs:
            validate_request_struct(r, n_tiles)
            # Count enhancement tiles in this request
            enh_tiles = [t for t in r['tiles'] if t['layer'] == 1]
            print(f"  User {r['u']} Video {r['video']} EnhTiles={len(enh_tiles)}")
        all_requests.extend(reqs)

        reqs = user_env.step(by_video=by_video)

    print('\nSummary:')
    print(f'Total requests collected: {len(all_requests)}')
    base_sizes = set(t['size'] for r in all_requests for t in r['tiles'] if t['layer']==0)
    enh_sizes = set(t['size'] for r in all_requests for t in r['tiles'] if t['layer']==1)
    print(f'Base layer size values: {base_sizes}')
    print(f'Enh layer size values: {enh_sizes}')

    print('\nSample request (first user, first step):')
    # Print all requests in a readable format
    print(json.dumps(all_requests, indent=4))

===== UserTileRequestEvents Test Harness =====
Grid: 4x4 -> 16 tiles | Users: 2 | Layers: 2
Running steps...
User arrivals reset: [1 5]
 Step 0: got 0 user request bundles
 Step 1: got 1 user request bundles
  User 0 Video 0 EnhTiles=4
 Step 2: got 1 user request bundles
  User 0 Video 0 EnhTiles=4

Summary:
Total requests collected: 2
Base layer size values: {125000.0}
Enh layer size values: {937500.0}

Sample request (first user, first step):
[
    {
        "seq": 1,
        "u": 0,
        "p": 0,
        "video": 0,
        "tiles": [
            {
                "tile": 0,
                "layer": 0,
                "size": 125000.0,
                "events": {
                    "alpha_p_u": 1,
                    "alpha_M_u": 0,
                    "alpha_C_u": 0,
                    "beta_p_u": 0,
                    "beta_M_u": 0
                }
            },
            {
                "tile": 1,
                "layer": 0,
                "size": 125000.0,
          